# CNTLib: StarDist Training

This notebook demonstrates StarDist training using the released CNT dataset and
an explicitly selected frozen train/validation/test split manifest.

The dataset preparation procedure is documented separately in
`01_dataset_preparation.ipynb`.

The released dataset already contains the split manifests used for the reported
experiments. This notebook therefore consumes an existing split manifest and
does not generate or modify dataset membership.

Workflow:

1. Configure the dataset, model, output, and split-manifest locations.
2. Validate the released dataset and selected frozen split.
3. Configure StarDist training.
4. Train the model.
5. Inspect the generated model and training artifacts.

## 1. Setup

In [1]:
from datetime import datetime
from pathlib import Path

import cnt_project

from cnt_project.io.paths import DatasetPaths, OutputPaths

from utils import run_module

### Verify CNTLib installation

The path printed below should point to the installed CNTLib package in the
current notebook environment.

In [2]:
print("CNTLib import package:")
print(cnt_project.__file__)

CNTLib import package:
C:\Users\abd93000\PycharmProjects\cnt_project_review\src\cnt_project\__init__.py


## 2. Configuration

CNTLib separates three important filesystem locations:

- `DATASET_ROOT` — canonical CNT dataset containing `images/`, `masks/`,
  metadata, split information, and COCO annotations;
- `MODEL_BASEDIR` — directory containing trained StarDist model folders;
- `GLOBAL_OUTPUTS_ROOT` — root under which CNTLib manages `runs/`,
  `reports/`, and `misc/`.

Only the root locations are configured by the consumer. CNTLib manages their
internal canonical structure.

In [3]:
# ------------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "cnt_segmentation"
)

MODEL_BASEDIR = (
    PROJECT_ROOT
    / "models"
)

GLOBAL_OUTPUTS_ROOT = (
    PROJECT_ROOT
    / "global_outputs"
)


# ------------------------------------------------------------------
# Frozen experimental split
# ------------------------------------------------------------------

SPLIT_NAME = "stratified_seed_0"


# ------------------------------------------------------------------
# Training identity
# ------------------------------------------------------------------

EXAMPLE_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

MODEL_NAME = (
    f"cntlib_notebook_training_example_{EXAMPLE_TIMESTAMP}"
)

RUN_NAME = (
    f"cntlib_notebook_training_example_{EXAMPLE_TIMESTAMP}"
)


# ------------------------------------------------------------------
# Smoke-test training configuration
# ------------------------------------------------------------------

MAX_TRAIN_SAMPLES = 3
MAX_VAL_SAMPLES = 2

EPOCHS = 1
STEPS_PER_EPOCH = 1

### resolve canonical paths

In [4]:
dataset_paths = DatasetPaths.from_root(
    DATASET_ROOT
)

output_paths = OutputPaths.from_root(
    GLOBAL_OUTPUTS_ROOT
)

DATASET_ROOT = dataset_paths.root
GLOBAL_OUTPUTS_ROOT = output_paths.root

SPLIT_MANIFEST = dataset_paths.split_manifest_csv(
    SPLIT_NAME
)

MODEL_DIR = (
    MODEL_BASEDIR.resolve()
    / MODEL_NAME
)

### Validate the initial dataset

Before preparation, the dataset must at minimum contain matching source
`images/` and `masks/` directories.

The preparation workflow generates or validates downstream artifacts such as:

- density metadata;
- noise metadata;
- train/validation/test split manifests;
- split metadata;
- COCO ground-truth annotations.

In [5]:
IMAGES_DIR = dataset_paths.images_root
MASKS_DIR = dataset_paths.masks_root

print("Dataset root:")
print(DATASET_ROOT)

print("\nImages directory:")
print(IMAGES_DIR)

print("\nMasks directory:")
print(MASKS_DIR)

print("\nModel base directory:")
print(MODEL_BASEDIR)

print("\nGlobal outputs root:")
print(GLOBAL_OUTPUTS_ROOT)


if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist: {DATASET_ROOT}"
    )

if not IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Dataset images directory does not exist: {IMAGES_DIR}"
    )

if not MASKS_DIR.exists():
    raise FileNotFoundError(
        f"Dataset masks directory does not exist: {MASKS_DIR}"
    )

image_files = [
    path
    for path in IMAGES_DIR.iterdir()
    if path.is_file()
]

mask_files = [
    path
    for path in MASKS_DIR.iterdir()
    if path.is_file()
]

print(f"\nImages found: {len(image_files)}")
print(f"Masks found:  {len(mask_files)}")

if not image_files:
    raise ValueError(
        "The dataset contains no image files."
    )

if not mask_files:
    raise ValueError(
        "The dataset contains no mask files."
    )

MODEL_BASEDIR.mkdir(
    parents=True,
    exist_ok=True,
)

output_paths.ensure()

print("\nInitial dataset configuration is valid.")

Dataset root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation

Images directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\images

Masks directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\masks

Model base directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\models

Global outputs root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs

Images found: 130
Masks found:  130

Initial dataset configuration is valid.


## 3. StarDist Training

The prepared split manifest can now be used by CNTLib's canonical StarDist
training runner.

The runner loads the manifest-defined training and validation subsets,
initializes or loads a StarDist model, performs training, and stores training
artifacts under the configured global output root.

This notebook uses a very small fresh-model configuration as a functional
example. It is not intended to produce a useful trained segmentation model.

In [6]:
run_module(
    "cnt_project.model_development.training.runners.train_stardist_runner",
    "--help",
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.model_development.training.runners.train_stardist_runner --help
--------------------------------------------------------------------------------
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-09-16 21:19:38.319877: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-16 21:19:40.527030: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from diffe

### Build the training configuration

This example uses:

- fresh model initialization;
- grayscale input;
- 32 StarDist rays;
- a `2 x 2` grid;
- MAE distance loss;
- three training samples;
- two validation samples;
- one epoch;
- one training step.

Post-training threshold optimization is skipped because this notebook is
testing the training workflow rather than performing a meaningful model
optimization experiment.

In [7]:
training_args = [
    "--dataset-root",
    str(DATASET_ROOT),

    "--split-manifest-path",
    str(SPLIT_MANIFEST),

    "--model-basedir",
    str(MODEL_BASEDIR),

    "--global-outputs-root",
    str(GLOBAL_OUTPUTS_ROOT),

    "--model-name",
    MODEL_NAME,

    "--run-name",
    RUN_NAME,

    "--initialization-mode",
    "fresh_config",

    "--grayscale",
    "true",

    "--n-rays",
    "32",

    "--grid",
    "2,2",

    "--distance-loss",
    "mae",

    "--epochs",
    str(EPOCHS),

    "--steps-per-epoch",
    str(STEPS_PER_EPOCH),

    "--max-train-samples",
    str(MAX_TRAIN_SAMPLES),

    "--max-val-samples",
    str(MAX_VAL_SAMPLES),

    "--skip-threshold-optimization",
]

training_args

['--dataset-root',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\data\\cnt_segmentation',
 '--split-manifest-path',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\data\\cnt_segmentation\\splits\\stratified_seed_0.csv',
 '--model-basedir',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\models',
 '--global-outputs-root',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\global_outputs',
 '--model-name',
 'cntlib_notebook_training_example_20260916_211924',
 '--run-name',
 'cntlib_notebook_training_example_20260916_211924',
 '--initialization-mode',
 'fresh_config',
 '--grayscale',
 'true',
 '--n-rays',
 '32',
 '--grid',
 '2,2',
 '--distance-loss',
 'mae',
 '--epochs',
 '1',
 '--steps-per-epoch',
 '1',
 '--max-train-samples',
 '3',
 '--max-val-samples',
 '2',
 '--skip-threshold-optimization']

### Run training

In [8]:
run_module(
    "cnt_project.model_development.training.runners.train_stardist_runner",
    *training_args,
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.model_development.training.runners.train_stardist_runner --dataset-root C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation --split-manifest-path C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\splits\stratified_seed_0.csv --model-basedir C:\Users\abd93000\PycharmProjects\cnt_project_review\models --global-outputs-root C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs --model-name cntlib_notebook_training_example_20260916_211924 --run-name cntlib_notebook_training_example_20260916_211924 --initialization-mode fresh_config --grayscale true --n-rays 32 --grid 2,2 --distance-loss mae --epochs 1 --steps-per-epoch 1 --max-train-samples 3 --max-val-samples 2 --skip-threshold-optimization
--------------------------------------------------------------------------------
bioimageio_utils.py (2): pkg_resources is deprec

## 4. Inspect Generated Training Artifacts

The trained StarDist model is stored under `MODEL_BASEDIR`.

Run-specific training metadata and diagnostic artifacts are stored beneath:

```text
<GLOBAL_OUTPUTS_ROOT>/
└── runs/
    └── <RUN_NAME>/
        ├── train/
        ├── inference/
        ├── eval/
        └── viz/
```

#### resolve output paths

In [9]:
RUN_ROOT = (
    GLOBAL_OUTPUTS_ROOT
    / "runs"
    / RUN_NAME
)

TRAIN_DIR = (
    RUN_ROOT
    / "train"
)

MODEL_DIR = (
    MODEL_BASEDIR.resolve()
    / MODEL_NAME
)

print("Model directory:")
print(MODEL_DIR)

print("\nRun root:")
print(RUN_ROOT)

print("\nTraining artifacts:")
print(TRAIN_DIR)

Model directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\models\cntlib_notebook_training_example_20260916_211924

Run root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_training_example_20260916_211924

Training artifacts:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_training_example_20260916_211924\train


### Validate final outputs

In [10]:
if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Trained model directory does not exist: {MODEL_DIR}"
    )

if not TRAIN_DIR.exists():
    raise FileNotFoundError(
        f"Training artifact directory does not exist: {TRAIN_DIR}"
    )

print("Training workflow completed successfully.")

Training workflow completed successfully.


### list artifacts

In [11]:
print("Model files:")

for path in sorted(MODEL_DIR.rglob("*")):
    if path.is_file():
        print(
            " -",
            path.relative_to(MODEL_DIR),
        )


print("\nRun files:")

for path in sorted(RUN_ROOT.rglob("*")):
    if path.is_file():
        print(
            " -",
            path.relative_to(RUN_ROOT),
        )

Model files:
 - config.json
 - weights_best.h5
 - weights_last.h5

Run files:
 - train\history.json
 - train\metrics\dist_dist_iou_metric.png
 - train\metrics\dist_loss.png
 - train\metrics\dist_relevant_mae.png
 - train\metrics\dist_relevant_mse.png
 - train\metrics\learning_rate.png
 - train\metrics\loss.png
 - train\metrics\prob_kld.png
 - train\metrics\prob_loss.png
 - train\metrics\val_dist_dist_iou_metric.png
 - train\metrics\val_dist_loss.png
 - train\metrics\val_dist_relevant_mae.png
 - train\metrics\val_dist_relevant_mse.png
 - train\metrics\val_loss.png
 - train\metrics\val_prob_kld.png
 - train\metrics\val_prob_loss.png
 - train\status.json
 - train\training_manifest.json


## 5. Next Step

Training is complete.

The trained model can now be used to generate predictions on the test subset.

Continue with:

`03_inference.ipynb`

Alternatively, the inference notebook can be used directly with a released
pretrained checkpoint without rerunning training.